# 🎨 Neural Style Transfer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TUSHARTAMRAKAR/neural-style-transfer/blob/main/notebook/nst_colab.ipynb)

This notebook implements **Neural Style Transfer** from the paper:
> *A Neural Algorithm of Artistic Style* — Gatys, Ecker & Bethge (2015)

### What you'll learn:
- How VGG-19 extracts content and style features
- What Gram matrices are and why they capture texture
- How L-BFGS optimization works on image pixels

### Runtime: Enable GPU for best results
`Runtime → Change runtime type → T4 GPU`

## Step 1: Install dependencies & check GPU

In [ ]:
# PyTorch + torchvision are pre-installed in Colab
# Just verify we have a GPU
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('✅ GPU detected — this will run fast!')
else:
    print('⚠️  No GPU — will run on CPU (~3-5 min per image)')

## Step 2: Upload your images

In [ ]:
# Option A: Upload from your computer
from google.colab import files
from IPython.display import display, Image as IPImage

print('Upload your CONTENT image (your photo):')
uploaded_content = files.upload()
content_path = list(uploaded_content.keys())[0]
print(f'Content: {content_path}')

print('\nUpload your STYLE image (the artwork):')
uploaded_style = files.upload()
style_path = list(uploaded_style.keys())[0]
print(f'Style: {style_path}')

In [ ]:
# Preview your images side by side
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(Image.open(content_path))
axes[0].set_title('Content Image (your photo)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(Image.open(style_path))
axes[1].set_title('Style Image (the artwork)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Step 3: Clone the NST engine from GitHub
We use our production `nst_engine.py` — same code powering the web app.

In [ ]:
!git clone https://github.com/TUSHARTAMRAKAR/neural-style-transfer.git /tmp/nst
import sys
sys.path.insert(0, '/tmp/nst/backend')

from nst_engine import run_style_transfer, get_style_presets
print('✅ NST engine loaded!')
print(f'Available presets: {list(get_style_presets().keys())}')

## Step 4: Understanding the algorithm

Before we run it, let's visualize **what VGG-19 actually sees**.

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load VGG-19
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(DEVICE).eval()
print('VGG-19 loaded!')

# Load content image
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
img = transform(Image.open(content_path).convert('RGB')).unsqueeze(0).to(DEVICE)

# Extract feature maps from early and deep layers
activations = {}
def hook(name):
    def fn(module, input, output): activations[name] = output.detach()
    return fn

vgg[1].register_forward_hook(hook('relu1_1'))   # early: edges
vgg[20].register_forward_hook(hook('relu4_1'))  # mid: shapes
vgg[22].register_forward_hook(hook('relu4_2'))  # content layer: objects

with torch.no_grad(): vgg(img)

# Visualize first 8 channels of each layer
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
layer_names = ['relu1_1 (edges/colors)', 'relu4_1 (shapes)', 'relu4_2 (objects)']

for row, (layer, name) in enumerate(zip(['relu1_1','relu4_1','relu4_2'], layer_names)):
    feat = activations[layer][0].cpu().numpy()
    axes[row, 0].set_ylabel(name, fontsize=8, rotation=90, labelpad=5)
    for col in range(8):
        if col < feat.shape[0]:
            axes[row, col].imshow(feat[col], cmap='viridis')
        axes[row, col].axis('off')

plt.suptitle('VGG-19 Feature Maps — What the network sees at different depths', fontsize=12)
plt.tight_layout()
plt.show()
print('Early layers detect edges. Deep layers detect objects and structure.')

In [ ]:
# Visualize the Gram matrix — the SECRET of style capture
# Gram matrix = channel correlations = texture statistics

def gram_matrix(features):
    b, c, h, w = features.size()
    f = features.view(b * c, h * w)
    return torch.mm(f, f.t()).div(b * c * h * w)

# Load style image
style_img = transform(Image.open(style_path).convert('RGB')).unsqueeze(0).to(DEVICE)
style_acts = {}
vgg[1].register_forward_hook(lambda m,i,o: style_acts.update({'relu1_1': o.detach()}))

with torch.no_grad(): vgg(style_img)

feat   = style_acts['relu1_1']       # [1, 64, H, W]
gram   = gram_matrix(feat).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(Image.open(style_path))
axes[0].set_title('Style image', fontweight='bold'); axes[0].axis('off')

im = axes[1].imshow(gram[:32,:32], cmap='plasma')
plt.colorbar(im, ax=axes[1])
axes[1].set_title('Gram matrix (32×32 subset)\nHigh value = these channels co-activate', fontweight='bold')

plt.suptitle('This 64×64 matrix captures ALL the texture info from the artwork', fontsize=11)
plt.tight_layout()
plt.show()

## Step 5: Run Neural Style Transfer 🚀

In [ ]:
# ── Configuration ──────────────────────────────────────────
OUTPUT_PATH    = 'stylized_result.png'
NUM_STEPS      = 300    # 100 = quick preview, 300 = good quality, 500 = best
CONTENT_WEIGHT = 1e4    # Higher = more photo-like
STYLE_WEIGHT   = 1e6    # Higher = more painterly
# ─────────────────────────────────────────────────────────────

steps_log = []

def log_progress(step, total, content_loss, style_loss, total_loss):
    steps_log.append({'step': step, 'total': total_loss})
    print(f'Step {step:3d}/{total} | Total loss: {total_loss:.2e}', end='\r')

print(f'Starting NST on {DEVICE}...')
print(f'Steps: {NUM_STEPS} | This will take ~{'30 sec' if torch.cuda.is_available() else '3-5 min'}\n')

result_path = run_style_transfer(
    content_path=content_path,
    style_path=style_path,
    output_path=OUTPUT_PATH,
    num_steps=NUM_STEPS,
    content_weight=CONTENT_WEIGHT,
    style_weight=STYLE_WEIGHT,
    progress_callback=log_progress,
)

print(f'\n\n✅ Done! Saved to: {result_path}')

In [ ]:
# ── Show the result ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(Image.open(content_path))
axes[0].set_title('Content image', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(Image.open(style_path))
axes[1].set_title('Style image', fontsize=13, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(Image.open(result_path))
axes[2].set_title('Stylized result ✨', fontsize=13, fontweight='bold', color='#6366f1')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Loss curve
if steps_log:
    fig2, ax = plt.subplots(figsize=(8, 3))
    ax.plot([s['step'] for s in steps_log], [s['total'] for s in steps_log], color='#6366f1')
    ax.set_xlabel('Step'); ax.set_ylabel('Total loss')
    ax.set_title('Loss curve — decreasing = optimization working')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Download your result!
files.download(result_path)
print('🎨 Downloading stylized image...')

## 🧠 Key Takeaways

| Concept | What it does |
|---|---|
| VGG-19 | Pre-trained CNN that extracts rich visual features |
| Content loss | MSE on deep feature maps → preserves structure |
| Style loss | MSE on Gram matrices → captures texture/color |
| L-BFGS | Quasi-Newton optimizer → smarter than SGD for NST |
| Gram matrix | Channel correlations → position-independent texture stats |

### Want to learn more?
- 📄 [Original paper (Gatys et al.)](https://arxiv.org/abs/1508.06576)
- 🌐 [Live web app](https://huggingface.co/spaces/TUSHARTAMRAKAR/neural-style-transfer)
- 💻 [Source code on GitHub](https://github.com/TUSHARTAMRAKAR/neural-style-transfer)